# CS781 Phase 2 — Legal NER (L-NER) with our Phase-1 models

**One run = all experiments.** The first cell lists the experiments (architecture + fine-tuning method); *Save & Run All* trains them one after another and writes a separate `submission.zip` for each, plus `phase2_all_submissions.zip` with everything.

**Kaggle setup:** GPU T4 (or P100), Internet **On**, secret `HF_TOKEN` (your Hugging Face token), and attach the dataset that holds the Phase-1 model folder (`config.json`, `pytorch_model.bin`, `legal_bpe_tokenizer.json`).

**Pipeline:** documents → our own BPE tokenizer with character offsets → BIO tag per token (25 tags) → Phase-1 backbone + tag head over 512-token windows (overlaps averaged) → BIO → exact character spans → strict macro-F1 on held-out training documents → `prediction.json` for the test fold.

Methods: `full` fine-tuning, `partial` (top-K layers), `lora`, `lorma` (multiplicative low-rank, compulsory), `adapter` (Houlsby bottleneck), `bitfit`, `prompt` (soft prompt tuning), `kd` (knowledge distillation into a smaller student).

In [ ]:
EXPERIMENTS = [
    dict(ARCH="hybrid_prefixlm",  METHOD="full",  LR=1e-4),
    dict(ARCH="hybrid_prefixlm",  METHOD="lorma", LR=1e-3),
    dict(ARCH="decoder_llama",    METHOD="full",  LR=1e-4),
    dict(ARCH="encdec_t5",        METHOD="partial", LR=2e-4, PARTIAL_TOP_K=6),
    dict(ARCH="efficient_retnet", METHOD="kd", KD_TEACHER_RUN="hybrid_prefixlm_full"),
    dict(ARCH="decoder_llama",    METHOD="adapter", LR=1e-3),
]
TIME_BUDGET_HOURS = 11.0

CONTEXT = None
HEAD = "linear"
DECODE_MODE = "word_vote"
REPORT_DECODE_MODES = ["lenient", "strict", "word_vote"]

MAX_LEN = 512
STRIDE = 384
SPLIT_PUNCT = True

EPOCHS = 25
PATIENCE = 5
BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
LR = None
HEAD_LR = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.06
GRAD_CLIP = 1.0
AMP = True
SEED = 42
LOG_EVERY = 50

CLASS_WEIGHTING = "none"
O_WEIGHT = 0.5
EMPTY_WINDOW_KEEP = 1.0

PARTIAL_TOP_K = 4
RANK, ALPHA, PEFT_DROPOUT = 16, 32, 0.05
PEFT_TARGET_FFN = True
ADAPTER_DIM = 64
N_PROMPT = 32
KD_TEACHER_RUN = None
KD_TEACHER_DIR = None
KD_KEEP_LAYERS = [0, 2, 4, 6, 8, 10]
KD_ALPHA, KD_TEMPERATURE = 0.5, 2.0

VAL_FRAC = 0.15
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
DATA_PREFIX = "Phase_2_LNER"
LOCAL_DATA_DIR = None
MODEL_DIR = None
SMOKE_TEST = False
SAVE_WEIGHTS = False

DEFAULT_CONTEXT = {"decoder_llama": "bidir", "efficient_retnet": "echo", "hybrid_prefixlm": "bidir", "encdec_t5": "bidir"}
DEFAULT_LR = {"full": 5e-5, "kd": 1e-4, "partial": 1e-4, "lora": 5e-4, "lorma": 5e-4, "adapter": 5e-4,
              "bitfit": 1e-3, "prompt": 1e-3}
BASE_C = {k: v for k, v in dict(globals()).items() if k.isupper() and not k.startswith("_")}

def experiment_config(overrides):
    c = dict(BASE_C, **overrides)
    c["CONTEXT"] = overrides.get("CONTEXT") or DEFAULT_CONTEXT[c["ARCH"]]
    c["LR"] = overrides.get("LR") or DEFAULT_LR[c["METHOD"]]
    c["RUN_NAME"] = overrides.get("RUN_NAME") or f"{c['ARCH']}_{c['METHOD']}"
    if c["CONTEXT"] == "echo":
        assert c["MAX_LEN"] <= 640, "echo doubles the sequence: MAX_LEN must be <= 640 (max_positions 1344)"
    return c

C = experiment_config(EXPERIMENTS[0])
for _i, _e in enumerate(EXPERIMENTS, 1):
    _c = experiment_config(_e)
    print(f"{_i}. {_c['RUN_NAME']:26s} arch={_c['ARCH']:16s} method={_c['METHOD']:7s} context={_c['CONTEXT']:6s} lr={_c['LR']}")

In [ ]:
import os, sys, json, glob, time, random, shutil, zipfile
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()

def hf_token():
    if os.environ.get("HF_TOKEN"): return os.environ["HF_TOKEN"]
    for name in ("HF_TOKEN", "HF"):
        try:
            from kaggle_secrets import UserSecretsClient
            tok = UserSecretsClient().get_secret(name)
            if tok: return tok.strip()
        except Exception:
            pass
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        return None

HF_TOKEN_VALUE = hf_token()
if HF_TOKEN_VALUE:
    os.environ["HF_TOKEN"] = HF_TOKEN_VALUE
    print("Hugging Face token found (length", len(HF_TOKEN_VALUE), ")")
else:
    print("WARNING: no Hugging Face token found. On Kaggle: Add-ons -> Secrets -> add 'HF_TOKEN' (or 'HF') "
          "and switch its toggle ON for this notebook; the course dataset is gated.")

def data_file(rel):
    roots = [LOCAL_DATA_DIR] if LOCAL_DATA_DIR else []
    for root in roots:
        p = os.path.join(root, rel)
        if os.path.exists(p): return p
    hits = glob.glob(f"/kaggle/input/**/{rel}", recursive=True)
    if hits: return hits[0]
    from huggingface_hub import hf_hub_download
    assert HF_TOKEN_VALUE, ("No Hugging Face token: add the Kaggle secret HF_TOKEN (or HF), switch it ON for this notebook "
                           "(Add-ons -> Secrets), then run again.")
    return hf_hub_download(HF_DATASET_REPO, f"{DATA_PREFIX}/{rel}", repo_type="dataset", token=HF_TOKEN_VALUE)

def find_model_dir(arch):
    for cfgp in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            if json.load(open(cfgp)).get("arch") == arch and os.path.exists(os.path.join(os.path.dirname(cfgp), "pytorch_model.bin")):
                return os.path.dirname(cfgp)
        except Exception:
            pass
    raise FileNotFoundError(f"no Phase-1 model folder for arch={arch} under /kaggle/input "
                            "(attach the dataset that contains config.json + pytorch_model.bin + legal_bpe_tokenizer.json)")

### Phase-1 code (unchanged): our tokenizer and the four model architectures

In [ ]:
import heapq
import json
import os
import re
import time
from collections import Counter, defaultdict

PRETOKENIZE_PATTERN = (
    r"'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+"
)
EOS_TOKEN = "<|endoftext|>"

class LegalBPETokenizer:
    def __init__(self, merges=None, name="legal-bpe"):
        self.name = name
        self.merges = [tuple(m) for m in (merges or [])]
        self._pattern = re.compile(PRETOKENIZE_PATTERN)
        self._build_tables()

    def _build_tables(self):
        self.ranks = {pair: i for i, pair in enumerate(self.merges)}
        self.vocab = {i: bytes([i]) for i in range(256)}
        for i, (a, b) in enumerate(self.merges):
            self.vocab[256 + i] = self.vocab[a] + self.vocab[b]
        self.eos_token = EOS_TOKEN
        self.eos_token_id = 256 + len(self.merges)
        self.pad_token = self.eos_token
        self.pad_token_id = self.eos_token_id
        self.model_max_length = 10**9
        self._cache = {}

    def __len__(self):
        return 256 + len(self.merges) + 1

    @property
    def vocab_size(self):
        return len(self)

    def pretokenize(self, text):
        return self._pattern.findall(text)

    @classmethod
    def train(cls, texts, vocab_size, name="legal-bpe", min_frequency=2, verbose=True):
        assert 258 <= vocab_size <= 65535, "vocab_size must fit in uint16"
        t0 = time.time()
        pattern = re.compile(PRETOKENIZE_PATTERN)

        word_counts = Counter()
        n_chars = 0
        for text in texts:
            n_chars += len(text)
            word_counts.update(pattern.findall(text))
        if verbose:
            print(f"[BPE] {n_chars:,} characters, {len(word_counts):,} unique words "
                  f"({time.time() - t0:.0f}s)")

        words = [list(w.encode("utf-8")) for w in word_counts]
        freqs = list(word_counts.values())
        del word_counts

        pair_counts = defaultdict(int)
        pair_where = defaultdict(set)
        for wi, (w, f) in enumerate(zip(words, freqs)):
            for a, b in zip(w, w[1:]):
                pair_counts[(a, b)] += f
                pair_where[(a, b)].add(wi)
        heap = [(-c, p) for p, c in pair_counts.items()]
        heapq.heapify(heap)

        n_merges = vocab_size - 257
        merges = []
        while len(merges) < n_merges and heap:
            neg_c, pair = heapq.heappop(heap)
            count = pair_counts.get(pair, 0)
            if count <= 0:
                continue
            if -neg_c != count:
                heapq.heappush(heap, (-count, pair))
                continue
            if count < min_frequency:
                if verbose:
                    print(f"[BPE] stopping early: best pair frequency {count} < {min_frequency}")
                break

            new_id = 256 + len(merges)
            merges.append(pair)
            a, b = pair
            changed = set()
            for wi in list(pair_where[pair]):
                w, f = words[wi], freqs[wi]
                for p in zip(w, w[1:]):
                    pair_counts[p] -= f
                    changed.add(p)
                new_w, i = [], 0
                while i < len(w):
                    if i + 1 < len(w) and w[i] == a and w[i + 1] == b:
                        new_w.append(new_id)
                        i += 2
                    else:
                        new_w.append(w[i])
                        i += 1
                words[wi] = new_w
                for p in zip(new_w, new_w[1:]):
                    pair_counts[p] += f
                    pair_where[p].add(wi)
                    changed.add(p)
            del pair_where[pair]
            pair_counts.pop(pair, None)
            changed.discard(pair)
            for p in changed:
                c = pair_counts.get(p, 0)
                if c > 0:
                    heapq.heappush(heap, (-c, p))
                else:
                    pair_counts.pop(p, None)
                    pair_where.pop(p, None)
            if verbose and (len(merges) % 2000 == 0):
                print(f"[BPE] {len(merges):,}/{n_merges:,} merges ({time.time() - t0:.0f}s)")

        tok = cls(merges, name=name)
        if verbose:
            print(f"[BPE] done: vocab size {len(tok):,} ({time.time() - t0:.0f}s)")
        return tok

    def _encode_word(self, word):
        ids = self._cache.get(word)
        if ids is not None:
            return ids
        ids = list(word.encode("utf-8"))
        ranks = self.ranks
        while len(ids) >= 2:
            best_rank, best_pair = None, None
            for p in zip(ids, ids[1:]):
                r = ranks.get(p)
                if r is not None and (best_rank is None or r < best_rank):
                    best_rank, best_pair = r, p
            if best_pair is None:
                break
            a, b = best_pair
            new_id = 256 + best_rank
            merged, i = [], 0
            while i < len(ids):
                if i + 1 < len(ids) and ids[i] == a and ids[i + 1] == b:
                    merged.append(new_id)
                    i += 2
                else:
                    merged.append(ids[i])
                    i += 1
            ids = merged
        if len(self._cache) > 2_000_000:
            self._cache.clear()
        self._cache[word] = ids
        return ids

    def encode(self, text, add_special_tokens=False):
        out = []
        for word in self._pattern.findall(text):
            out.extend(self._encode_word(word))
        return out

    def __call__(self, texts, add_special_tokens=False, **kwargs):
        if isinstance(texts, str):
            return {"input_ids": self.encode(texts)}
        return {"input_ids": [self.encode(t) for t in texts]}

    def decode(self, ids, skip_special_tokens=True, **kwargs):
        parts = []
        for i in ids:
            i = int(i)
            if i == self.eos_token_id:
                if not skip_special_tokens:
                    parts.append(self.eos_token.encode("utf-8"))
                continue
            b = self.vocab.get(i)
            if b is not None:
                parts.append(b)
        return b"".join(parts).decode("utf-8", errors="replace")

    def batch_decode(self, sequences, skip_special_tokens=True, **kwargs):
        return [self.decode(s, skip_special_tokens=skip_special_tokens) for s in sequences]

    def convert_ids_to_tokens(self, ids):
        return [self.vocab[i].decode("utf-8", errors="replace") if i in self.vocab else self.eos_token
                for i in ids]

    def save_pretrained(self, directory):
        os.makedirs(directory, exist_ok=True)
        path = os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path, "w") as f:
            json.dump({
                "type": "byte-level-bpe",
                "name": self.name,
                "pattern": PRETOKENIZE_PATTERN,
                "eos_token": self.eos_token,
                "eos_token_id": self.eos_token_id,
                "vocab_size": len(self),
                "merges": [list(m) for m in self.merges],
            }, f)
        return path

    @classmethod
    def from_pretrained(cls, directory):
        path = directory if directory.endswith(".json") else os.path.join(directory, "legal_bpe_tokenizer.json")
        with open(path) as f:
            data = json.load(f)
        assert data["pattern"] == PRETOKENIZE_PATTERN, "tokenizer was saved with a different pre-tokenisation pattern"
        tok = cls(data["merges"], name=data.get("name", "legal-bpe"))
        assert tok.eos_token_id == data["eos_token_id"]
        return tok

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        xf = xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)
        return (xf * self.weight.float()).type_as(x)

class RotaryEmbedding(nn.Module):
    def __init__(self, head_dim, max_positions, theta=10_000.0):
        super().__init__()
        inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2, dtype=torch.float32) / head_dim))
        freqs = torch.outer(torch.arange(max_positions, dtype=torch.float32), inv_freq)
        self.register_buffer("cos_cached", torch.cat([freqs.cos(), freqs.cos()], dim=-1), persistent=False)
        self.register_buffer("sin_cached", torch.cat([freqs.sin(), freqs.sin()], dim=-1), persistent=False)

    def forward(self, x, offset=0):
        T = x.size(-2)
        cos = self.cos_cached[offset:offset + T].to(x.dtype)
        sin = self.sin_cached[offset:offset + T].to(x.dtype)
        x1, x2 = x.chunk(2, dim=-1)
        return x * cos + torch.cat((-x2, x1), dim=-1) * sin

class SelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        B, T, C = x.shape
        q, k, v = self.qkv(x).view(B, T, 3, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k = rope(q, offset), rope(k, offset)
        if cache is not None:
            if "k" in cache:
                k = torch.cat([cache["k"], k], dim=2)
                v = torch.cat([cache["v"], v], dim=2)
            cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            is_causal=is_causal and attn_mask is None,
            dropout_p=self.dropout if self.training else 0.0,
        )
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class SwiGLU(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_gate_up = nn.Linear(n_embd, 2 * hidden, bias=False)
        self.w_down = nn.Linear(hidden, n_embd, bias=False)

    def forward(self, x):
        gate, up = self.w_gate_up(x).chunk(2, dim=-1)
        return self.w_down(F.silu(gate) * up)

class DecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, attn_mask=None, is_causal=True, cache=None, offset=0):
        x = x + self.drop(self.attn(self.norm1(x), rope, attn_mask, is_causal, cache, offset))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

def init_weights(model, n_layers, std=0.02):
    for module in model.modules():
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=std)
    for name, p in model.named_parameters():
        if name.endswith("proj.weight") or name.endswith("w_down.weight"):
            nn.init.normal_(p, mean=0.0, std=std / math.sqrt(2 * n_layers))

def lm_loss(lm_head, hidden, input_ids, loss_start=0):
    start = max(int(loss_start), 1)
    logits = lm_head(hidden[:, start - 1:-1])
    targets = input_ids[:, start:]
    return F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))

class DecoderOnlyLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        h = self._hidden(ids)
        return self.lm_head(h[:, prompt_ids.size(1) - 1:-1])

def prefix_lm_mask(T, prefix_len, device):
    i = torch.arange(T, device=device)
    mask = (i[None, :] <= i[:, None]) | (i[None, :] < prefix_len)
    return mask[None, None]

class PrefixLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.segment = nn.Embedding(2, d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList(
            [DecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)]
        )
        self.norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, prefix_len, caches=None, offset=0, decoding=False):
        B, T = input_ids.shape
        positions = torch.arange(offset, offset + T, device=input_ids.device)
        segments = (positions >= prefix_len).long()
        x = self.drop(self.embed(input_ids) + self.segment(segments)[None])

        if decoding:
            attn_mask, is_causal = None, False
        elif prefix_len >= T:
            attn_mask, is_causal = None, False
        elif prefix_len <= 0:
            attn_mask, is_causal = None, True
        else:
            attn_mask, is_causal = prefix_lm_mask(T, prefix_len, input_ids.device), False

        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, attn_mask=attn_mask, is_causal=is_causal,
                      cache=None if caches is None else caches[i], offset=offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        return lm_loss(self.lm_head, self._hidden(input_ids, P), input_ids, P)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        P = prompt_ids.size(1)
        caches = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, prefix_len=P, caches=caches, offset=0)
        return self.lm_head(h[:, -1]), {"caches": caches, "pos": P, "prefix_len": P}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["prefix_len"], state["caches"],
                         offset=state["pos"], decoding=True)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        P = prompt_ids.size(1)
        h = self._hidden(torch.cat([prompt_ids, cont_ids], dim=1), prefix_len=P)
        return self.lm_head(h[:, P - 1:-1])

class CrossAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout=0.0):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.q = nn.Linear(n_embd, n_embd, bias=False)
        self.kv = nn.Linear(n_embd, 2 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout

    def forward(self, x, memory, cache=None):
        B, T, C = x.shape
        q = self.q(x).view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        if cache is not None and "k" in cache:
            k, v = cache["k"], cache["v"]
        else:
            S = memory.size(1)
            k, v = self.kv(memory).view(B, S, 2, self.n_head, self.head_dim).permute(2, 0, 3, 1, 4)
            if cache is not None:
                cache["k"], cache["v"] = k, v
        y = F.scaled_dot_product_attention(q, k, v, dropout_p=self.dropout if self.training else 0.0)
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class EncoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope):
        x = x + self.drop(self.attn(self.norm1(x), rope, is_causal=False))
        x = x + self.drop(self.mlp(self.norm2(x)))
        return x

class CrossDecoderBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.self_attn = SelfAttention(n_embd, n_head, dropout)
        self.norm2 = RMSNorm(n_embd)
        self.cross_attn = CrossAttention(n_embd, n_head, dropout)
        self.norm3 = RMSNorm(n_embd)
        self.mlp = SwiGLU(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, memory, rope, cache=None, offset=0, is_causal=True):
        self_cache = None if cache is None else cache["self"]
        cross_cache = None if cache is None else cache["cross"]
        x = x + self.drop(self.self_attn(self.norm1(x), rope, is_causal=is_causal, cache=self_cache, offset=offset))
        x = x + self.drop(self.cross_attn(self.norm2(x), memory, cross_cache))
        x = x + self.drop(self.mlp(self.norm3(x)))
        return x

class EncoderDecoderLM(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, H = cfg["n_embd"], cfg["n_head"]
        self.bos_id = cfg["bos_token_id"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.enc_blocks = nn.ModuleList(
            [EncoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_enc_layer"])]
        )
        self.enc_norm = RMSNorm(d)
        self.dec_blocks = nn.ModuleList(
            [CrossDecoderBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(cfg["n_dec_layer"])]
        )
        self.dec_norm = RMSNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, cfg["n_enc_layer"] + cfg["n_dec_layer"])

    def encode(self, src_ids):
        x = self.drop(self.embed(src_ids))
        for block in self.enc_blocks:
            x = block(x, self.rope)
        return self.enc_norm(x)

    def decode(self, tgt_in, memory, caches=None, offset=0, is_causal=True):
        x = self.drop(self.embed(tgt_in))
        for i, block in enumerate(self.dec_blocks):
            x = block(x, memory, self.rope, None if caches is None else caches[i], offset, is_causal)
        return self.dec_norm(x)

    def _shift_right(self, tgt):
        bos = torch.full_like(tgt[:, :1], self.bos_id)
        return torch.cat([bos, tgt[:, :-1]], dim=1)

    def forward(self, input_ids, prefix_len):
        P = int(prefix_len)
        assert 0 < P < input_ids.size(1), "encoder-decoder needs 0 < prefix_len < block size"
        src, tgt = input_ids[:, :P], input_ids[:, P:]
        h = self.decode(self._shift_right(tgt), self.encode(src))
        logits = self.lm_head(h)
        return F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1))

    @torch.no_grad()
    def prefill(self, prompt_ids):
        memory = self.encode(prompt_ids)
        caches = [{"self": {}, "cross": {}} for _ in self.dec_blocks]
        bos = torch.full((prompt_ids.size(0), 1), self.bos_id, dtype=torch.long, device=prompt_ids.device)
        h = self.decode(bos, memory, caches, offset=0, is_causal=True)
        return self.lm_head(h[:, -1]), {"memory": memory, "caches": caches, "pos": 1}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self.decode(token_ids[:, None], state["memory"], state["caches"], offset=state["pos"], is_causal=False)
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        return self.lm_head(self.decode(self._shift_right(cont_ids), self.encode(prompt_ids)))

class MultiScaleRetention(nn.Module):

    def __init__(self, n_embd, n_head):
        super().__init__()
        self.n_head, self.head_dim = n_head, n_embd // n_head
        self.qkvg = nn.Linear(n_embd, 4 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        gammas = 1.0 - 2.0 ** (-5.0 - torch.arange(n_head, dtype=torch.float32))
        self.register_buffer("log_gamma", torch.log(gammas), persistent=False)

    def _decay_matrix(self, T, device):
        idx = torch.arange(T, device=device, dtype=torch.float32)
        diff = idx[:, None] - idx[None, :]
        decay = torch.exp(diff.clamp(min=0)[None] * self.log_gamma[:, None, None])
        decay = decay.masked_fill(diff[None] < 0, 0.0)
        return decay / decay.sum(dim=-1, keepdim=True).sqrt()

    def forward(self, x, rope, state=None, offset=0):
        B, T, C = x.shape
        q, k, v, g = self.qkvg(x).split(C, dim=-1)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        q, k = rope(q, offset), rope(k, offset)
        k = k * self.head_dim ** -0.5

        if state is not None and "S" in state:
            with torch.autocast(device_type=x.device.type, enabled=False):
                gamma = self.log_gamma.exp().view(1, -1, 1, 1)
                state["S"] = gamma * state["S"] + k.float().transpose(-1, -2) @ v.float()
                state["scale"] = gamma * state["scale"] + 1.0
                y = ((q.float() @ state["S"]) / state["scale"].sqrt()).to(v.dtype)
        else:
            decay = self._decay_matrix(T, x.device).to(q.dtype)
            y = ((q @ k.transpose(-1, -2)) * decay) @ v
            if state is not None:
                with torch.autocast(device_type=x.device.type, enabled=False):
                    pos = torch.arange(T, device=x.device, dtype=torch.float32)
                    w = torch.exp((T - 1 - pos)[None, :] * self.log_gamma[:, None])
                    state["S"] = torch.einsum("bhtd,ht,bhte->bhde", k.float(), w, v.float())
                    state["scale"] = w.sum(dim=-1).view(1, -1, 1, 1)

        y = y.transpose(1, 2).float()
        y = y * torch.rsqrt(y.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
        y = y.reshape(B, T, C).to(g.dtype)
        return self.proj(F.silu(g) * y)

class FeedForward(nn.Module):
    def __init__(self, n_embd, hidden):
        super().__init__()
        self.w_in = nn.Linear(n_embd, hidden)
        self.w_down = nn.Linear(hidden, n_embd)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_in(x)))

class RetNetBlock(nn.Module):
    def __init__(self, n_embd, n_head, ffn_hidden, dropout=0.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(n_embd)
        self.retention = MultiScaleRetention(n_embd, n_head)
        self.norm2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, ffn_hidden)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, rope, state=None, offset=0):
        x = x + self.drop(self.retention(self.norm1(x), rope, state, offset))
        x = x + self.drop(self.ffn(self.norm2(x)))
        return x

class RetNetLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        d, L, H = cfg["n_embd"], cfg["n_layer"], cfg["n_head"]
        self.embed = nn.Embedding(cfg["vocab_size"], d)
        self.drop = nn.Dropout(cfg["dropout"])
        self.rope = RotaryEmbedding(d // H, cfg["max_positions"], cfg["rope_theta"])
        self.blocks = nn.ModuleList([RetNetBlock(d, H, cfg["ffn_hidden"], cfg["dropout"]) for _ in range(L)])
        self.norm = nn.LayerNorm(d)
        self.lm_head = nn.Linear(d, cfg["vocab_size"], bias=False)
        self.lm_head.weight = self.embed.weight
        init_weights(self, L)

    def _hidden(self, input_ids, states=None, offset=0):
        x = self.drop(self.embed(input_ids))
        for i, block in enumerate(self.blocks):
            x = block(x, self.rope, None if states is None else states[i], offset)
        return self.norm(x)

    def forward(self, input_ids, prefix_len=0):
        return lm_loss(self.lm_head, self._hidden(input_ids), input_ids, prefix_len)

    @torch.no_grad()
    def prefill(self, prompt_ids):
        states = [dict() for _ in self.blocks]
        h = self._hidden(prompt_ids, states, offset=0)
        return self.lm_head(h[:, -1]), {"states": states, "pos": prompt_ids.size(1)}

    @torch.no_grad()
    def decode_step(self, token_ids, state):
        h = self._hidden(token_ids[:, None], state["states"], offset=state["pos"])
        state["pos"] += 1
        return self.lm_head(h[:, -1]), state

    @torch.no_grad()
    def teacher_forced_logits(self, prompt_ids, cont_ids):
        ids = torch.cat([prompt_ids, cont_ids], dim=1)
        return self.lm_head(self._hidden(ids)[:, prompt_ids.size(1) - 1:-1])

ARCH_CLASSES = {'decoder_llama': DecoderOnlyLM, 'hybrid_prefixlm': PrefixLM, 'encdec_t5': EncoderDecoderLM, 'efficient_retnet': RetNetLM}

### Phase-2 code: alignment, labels, decoding, scorer · NER model + adaptation methods · training

In [ ]:
import json, os, re, random
from collections import Counter, defaultdict

def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def parse_texts(obj):
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            out[str(k)] = v if isinstance(v, str) else (v.get("text") or "")
        return out
    out = {}
    for r in obj:
        did = r.get("id", r.get("doc_id", r.get("document_id")))
        out[str(did)] = r.get("text", "")
    return out

def _parse_entity(e):
    if isinstance(e, (list, tuple)):
        s, t, lab = int(e[0]), int(e[1]), str(e[2])
        txt = e[3] if len(e) > 3 else None
        return s, t, lab, txt
    s = e.get("start", e.get("start_char", e.get("begin")))
    t = e.get("end", e.get("end_char"))
    lab = e.get("label", e.get("type", e.get("entity")))
    return int(s), int(t), str(lab), e.get("text", e.get("entity_text"))

def parse_gold(obj):
    if isinstance(obj, dict):
        items = obj.items()
    else:
        items = ((r.get("id", r.get("doc_id")), r.get("entities", r.get("labels", []))) for r in obj)
    return {str(k): [_parse_entity(e) for e in v] for k, v in items}

def check_gold(texts, gold, labels, verbose=True):
    st = Counter()
    for did, ents in gold.items():
        text = texts.get(did)
        if text is None:
            st["doc_missing_text"] += 1; continue
        spans = sorted((s, e) for s, e, _, _ in ents)
        for (s1, e1), (s2, e2) in zip(spans, spans[1:]):
            if s2 < e1: st["overlapping_pairs"] += 1
        for s, e, lab, txt in ents:
            st["entities"] += 1
            if not (0 <= s < e <= len(text)): st["out_of_range"] += 1
            if lab not in labels: st["unknown_label"] += 1
            if txt is not None:
                if text[s:e] == txt: st["text_match"] += 1
                elif text[s:e].strip() == str(txt).strip(): st["text_match_after_strip"] += 1
                else: st["text_mismatch"] += 1
            if text[s:e] != text[s:e].strip(): st["span_has_outer_whitespace"] += 1
    if verbose:
        print("annotation check:", dict(st))
    return st

def split_docs(doc_ids, val_frac, seed):
    ids = sorted(doc_ids)
    random.Random(seed).shuffle(ids)
    n_val = max(1, int(round(len(ids) * val_frac))) if val_frac > 0 else 0
    return ids[n_val:], ids[:n_val]

_PUNCT_RUN = re.compile(r"^( ?)((?:[^\s\w]|_)+)$")

def tokenize_with_offsets(tok, text, split_punct=True):
    ids, starts, ends = [], [], []
    for m in tok._pattern.finditer(text):
        word, w0 = m.group(0), m.start()
        pieces = [(word, w0)]
        if split_punct:
            pm = _PUNCT_RUN.match(word)
            if pm and len(pm.group(2)) > 1:
                lead = pm.group(1)
                pieces = [((lead if j == 0 else "") + ch, w0 + (0 if j == 0 else len(lead) + j))
                          for j, ch in enumerate(pm.group(2))]
        for piece, p0 in pieces:
            b2c = []
            for ci, ch in enumerate(piece):
                b2c.extend([ci] * len(ch.encode("utf-8")))
            b2c.append(len(piece))
            pos = 0
            for tid in tok._encode_word(piece):
                nb = len(tok.vocab[tid])
                c0 = b2c[pos] if pos == 0 or b2c[pos] != b2c[pos - 1] else b2c[pos] + 1
                c1 = b2c[pos + nb] if pos + nb < len(b2c) - 1 else len(piece)
                if pos + nb < len(b2c) - 1 and b2c[pos + nb] == b2c[pos + nb - 1]:
                    c1 = b2c[pos + nb] + 1
                c0 = min(c0, c1)
                seg = piece[c0:c1]
                core = seg.strip()
                if core:
                    s = p0 + c0 + (len(seg) - len(seg.lstrip()))
                    e = s + len(core)
                else:
                    s = e = p0 + c0
                ids.append(tid); starts.append(s); ends.append(e)
                pos += nb
    return ids, starts, ends

def make_label_maps(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}

def spans_to_bio(starts, ends, ents, tag2id):
    n = len(starts)
    tags = [0] * n
    owner = [None] * n
    dropped = 0
    for s, e, lab, _ in sorted(ents, key=lambda x: -(x[1] - x[0])):
        idx = [i for i in range(n) if ends[i] > starts[i] and starts[i] < e and ends[i] > s]
        if not idx or any(owner[i] is not None for i in idx):
            dropped += 1; continue
        for j, i in enumerate(idx):
            owner[i] = (s, e)
            tags[i] = tag2id[("B-" if j == 0 else "I-") + lab]
    return tags, dropped

def bio_to_spans(tag_ids, starts, ends, id2tag, mode="lenient"):
    spans, cur = [], None
    for i, t in enumerate(tag_ids):
        if ends[i] <= starts[i]:
            continue
        tag = id2tag[int(t)]
        if tag == "O":
            if cur: spans.append(cur); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[2] != lab:
            if cur: spans.append(cur)
            cur = [starts[i], ends[i], lab] if (p == "B" or mode == "lenient") else None
        else:
            cur[1] = ends[i]
    if cur: spans.append(cur)
    return spans

def strict_scores(gold, pred, labels):
    tp, fp, fn = Counter(), Counter(), Counter()
    for did in gold:
        g = {(int(s), int(e), str(l)) for s, e, l, *_ in gold[did]}
        p = {(int(s), int(e), str(l)) for s, e, l, *_ in pred.get(did, [])}
        for x in g & p: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    per = {}
    for l in labels:
        if tp[l] + fp[l] + fn[l] == 0: continue
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        per[l] = {"P": P, "R": R, "F1": F, "support": tp[l] + fn[l], "pred": tp[l] + fp[l]}
    k = max(1, len(per))
    macro = {m: sum(v[m] for v in per.values()) / k for m in ("P", "R", "F1")}
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro_P = T / (T + FP) if T + FP else 0.0
    micro_R = T / (T + FN) if T + FN else 0.0
    micro_F = 2 * micro_P * micro_R / (micro_P + micro_R) if micro_P + micro_R else 0.0
    return {"macro_F1": macro["F1"], "macro_P": macro["P"], "macro_R": macro["R"],
            "micro_F1": micro_F, "per_label": per}

def make_windows(n_tokens, max_len, stride):
    if n_tokens <= max_len:
        return [(0, n_tokens)]
    out, s = [], 0
    while True:
        e = min(s + max_len, n_tokens)
        out.append((s, e))
        if e == n_tokens: break
        s += stride
    return out

def error_breakdown(gold, pred):
    out = Counter(); conf = Counter()
    for did in gold:
        g = [(int(s), int(e), l) for s, e, l, *_ in gold[did]]
        p = [(int(s), int(e), l) for s, e, l, *_ in pred.get(did, [])]
        ps = set(p)
        for s, e, l in g:
            if (s, e, l) in ps: out["correct"] += 1; continue
            ov = [x for x in p if x[0] < e and x[1] > s]
            if any(x[0] == s and x[1] == e for x in ov):
                out["wrong_label"] += 1
                conf[(l, [x for x in ov if x[0] == s and x[1] == e][0][2])] += 1
            elif any(x[2] == l for x in ov): out["boundary"] += 1
            elif ov: out["boundary+label"] += 1
            else: out["missed"] += 1
        gs = set(g)
        for s, e, l in p:
            if (s, e, l) in gs: continue
            if not any(x[0] < e and x[1] > s for x in g): out["spurious"] += 1
    return dict(out), {f"{a}->{b}": n for (a, b), n in conf.most_common(15)}

In [ ]:
import math, time, json, os
import torch
import torch.nn as nn
import torch.nn.functional as F

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.in_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

class LoRMALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.base, self.r, self.scale = base, r, alpha / r
        self.A = nn.Parameter(torch.empty(r, base.out_features))
        self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        y = self.base(x)
        return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale

class AdapterLinear(nn.Module):
    def __init__(self, base, bottleneck=64):
        super().__init__()
        self.base = base
        self.down = nn.Linear(base.out_features, bottleneck)
        self.up = nn.Linear(bottleneck, base.out_features)
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x):
        y = self.base(x)
        return y + self.up(F.gelu(self.down(y)))

class BiasLinear(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.bias = nn.Parameter(torch.zeros(base.out_features))
    def forward(self, x):
        return self.base(x) + self.bias

ATTN_LINEARS = ("qkv", "proj", "qkvg", "q", "kv")
FFN_LINEARS = ("w_gate_up", "w_down", "w_in")
OUT_PROJ = ("proj", "w_down")

def _replace_linears(module, names, make):
    n = 0
    for parent in list(module.modules()):
        for cname, child in list(parent.named_children()):
            if isinstance(child, nn.Linear) and cname in names:
                setattr(parent, cname, make(child)); n += 1
    return n

def backbone_blocks(arch, bb):
    return list(bb.enc_blocks) if arch == "encdec_t5" else list(bb.blocks)

class NERModel(nn.Module):
    def __init__(self, arch, backbone, n_tags, context="bidir", head="linear", n_prompt=0, dropout=0.1):
        super().__init__()
        self.arch, self.bb, self.context = arch, backbone, context
        self.head_out = n_tags
        d = backbone.cfg["n_embd"]
        self.drop = nn.Dropout(dropout)
        if head == "mlp":
            self.head = nn.Sequential(nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, n_tags))
        else:
            self.head = nn.Linear(d, n_tags)
        self.prompt = nn.Parameter(torch.randn(n_prompt, d) * 0.02) if n_prompt > 0 else None
        if hasattr(backbone, "lm_head"):
            backbone.lm_head = nn.Identity()
        if arch == "encdec_t5":
            backbone.dec_blocks = nn.ModuleList(); backbone.dec_norm = nn.Identity()

    def _mask(self, key_ok, T, causal):
        m = key_ok[:, None, None, :].expand(-1, 1, T, T)
        if causal:
            m = m & torch.ones(T, T, dtype=torch.bool, device=key_ok.device).tril()[None, None]
        eye = torch.eye(T, dtype=torch.bool, device=key_ok.device)[None, None]
        return m | eye

    def hidden(self, ids, attn):
        bb, arch = self.bb, self.arch
        B, T = ids.shape
        x = bb.embed(ids)
        ok = attn.bool()
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = torch.cat([x, x], dim=1); ok = torch.cat([ok, ok], dim=1)
        if arch == "hybrid_prefixlm":
            x = x + bb.segment(torch.zeros(x.size(1), dtype=torch.long, device=x.device))[None]
        P = 0
        if self.prompt is not None:
            P = self.prompt.size(0)
            x = torch.cat([self.prompt[None].expand(B, -1, -1).to(x.dtype), x], dim=1)
            ok = torch.cat([torch.ones(B, P, dtype=torch.bool, device=x.device), ok], dim=1)
        x = bb.drop(x)
        L = x.size(1)
        if arch == "efficient_retnet":
            for blk in bb.blocks:
                x = blk(x, bb.rope)
            x = bb.norm(x)
        elif arch == "encdec_t5":
            mask = self._mask(ok, L, causal=False)
            for blk in bb.enc_blocks:
                x = x + blk.drop(blk.attn(blk.norm1(x), bb.rope, attn_mask=mask, is_causal=False))
                x = x + blk.drop(blk.mlp(blk.norm2(x)))
            x = bb.enc_norm(x)
        else:
            causal = self.context in ("causal", "echo") and arch == "decoder_llama"
            mask = self._mask(ok, L, causal=causal)
            for blk in bb.blocks:
                x = blk(x, bb.rope, attn_mask=mask, is_causal=False)
            x = bb.norm(x)
        x = x[:, P:]
        if self.context == "echo" and arch in ("decoder_llama", "efficient_retnet"):
            x = x[:, T:]
        return x

    def forward(self, ids, attn):
        return self.head(self.drop(self.hidden(ids, attn)))

def load_backbone(model_dir, arch_classes, map_location="cpu"):
    cfg = json.load(open(os.path.join(model_dir, "config.json")))
    bb = arch_classes[cfg["arch"]](cfg)
    sd = torch.load(os.path.join(model_dir, "pytorch_model.bin"), map_location=map_location)
    missing, unexpected = bb.load_state_dict(sd, strict=False)
    missing = [k for k in missing if not k.endswith("lm_head.weight")]
    assert not missing and not unexpected, (missing, unexpected)
    return cfg, bb

def keep_layers(arch, bb, keep):
    if arch == "encdec_t5":
        bb.enc_blocks = nn.ModuleList([bb.enc_blocks[i] for i in keep]); bb.cfg["n_enc_layer"] = len(keep)
    else:
        bb.blocks = nn.ModuleList([bb.blocks[i] for i in keep]); bb.cfg["n_layer"] = len(keep)
    return bb

def apply_method(model, method, C):
    arch, bb = model.arch, model.bb
    info = {}
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "partial":
            blocks = backbone_blocks(arch, bb)
            for blk in blocks[-C["PARTIAL_TOP_K"]:]:
                for p in blk.parameters(): p.requires_grad = True
            norm = bb.enc_norm if arch == "encdec_t5" else bb.norm
            for p in norm.parameters(): p.requires_grad = True
        elif method in ("lora", "lorma"):
            cls = LoRALinear if method == "lora" else LoRMALinear
            targets = ATTN_LINEARS + (FFN_LINEARS if C["PEFT_TARGET_FFN"] else ())
            info["wrapped"] = _replace_linears(bb, targets, lambda l: cls(l, C["RANK"], C["ALPHA"], C["PEFT_DROPOUT"]))
        elif method == "adapter":
            info["wrapped"] = _replace_linears(bb, OUT_PROJ, lambda l: AdapterLinear(l, C["ADAPTER_DIM"]))
        elif method == "bitfit":
            info["wrapped"] = _replace_linears(bb, ATTN_LINEARS + FFN_LINEARS, BiasLinear)
            for n, p in bb.named_parameters():
                if n.endswith("bias") or "norm" in n: p.requires_grad = True
        elif method == "prompt":
            pass
        else:
            raise ValueError(method)
        for p in model.head.parameters(): p.requires_grad = True
        if model.prompt is not None: model.prompt.requires_grad = True
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_total = sum(p.numel() for p in model.parameters())
    info.update(trainable_params=n_train, total_params=n_total, trainable_pct=100.0 * n_train / n_total)
    return info

In [ ]:
import math, time, random
import numpy as np
import torch
import torch.nn.functional as F

def featurize(doc_ids, texts, gold, tok, tag2id, C):
    feats, dropped, n_ent = {}, 0, 0
    for did in doc_ids:
        ids, st, en = tokenize_with_offsets(tok, texts[did], split_punct=C["SPLIT_PUNCT"])
        f = {"ids": ids, "starts": st, "ends": en, "text": texts[did]}
        if gold is not None and did in gold:
            tags, d = spans_to_bio(st, en, gold[did], tag2id)
            f["tags"] = tags; dropped += d; n_ent += len(gold[did])
        feats[did] = f
    return feats, dropped, n_ent

def window_examples(feats, C, with_tags=True):
    ex = []
    for did, f in feats.items():
        for s, e in make_windows(len(f["ids"]), C["MAX_LEN"], C["STRIDE"]):
            item = {"doc": did, "s": s, "ids": f["ids"][s:e]}
            if with_tags: item["tags"] = f["tags"][s:e]
            ex.append(item)
    return ex

def collate(batch, pad_id, device):
    T = max(len(b["ids"]) for b in batch)
    ids = torch.full((len(batch), T), pad_id, dtype=torch.long)
    att = torch.zeros((len(batch), T), dtype=torch.long)
    tags = torch.full((len(batch), T), -100, dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["ids"])
        ids[i, :n] = torch.tensor(b["ids"]); att[i, :n] = 1
        if "tags" in b: tags[i, :n] = torch.tensor(b["tags"])
    return ids.to(device), att.to(device), tags.to(device)

@torch.no_grad()
def predict_logits(model, feats, C, pad_id, device):
    model.eval()
    out = {}
    ex = window_examples(feats, C, with_tags=False)
    acc = {d: torch.zeros(len(f["ids"]), model.head_out, dtype=torch.float32) for d, f in feats.items()}
    cnt = {d: torch.zeros(len(f["ids"]), 1) for d, f in feats.items()}
    bs = C["EVAL_BATCH_SIZE"]
    for i in range(0, len(ex), bs):
        batch = ex[i:i + bs]
        ids, att, _ = collate(batch, pad_id, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=C["AMP"] and device.type == "cuda"):
            logits = model(ids, att).float().cpu()
        for b, item in enumerate(batch):
            n = len(item["ids"])
            w = torch.ones(n, 1)
            acc[item["doc"]][item["s"]:item["s"] + n] += logits[b, :n] * w
            cnt[item["doc"]][item["s"]:item["s"] + n] += w
    for d in feats:
        out[d] = acc[d] / cnt[d].clamp(min=1)
    return out

def word_groups(starts, ends, text):
    groups, cur = [], []
    for i, (s, e) in enumerate(zip(starts, ends)):
        if e <= s:
            if cur: groups.append(cur); cur = []
            continue
        alnum = text[s].isalnum()
        if cur and alnum and starts[i] == ends[cur[-1]] and text[starts[cur[-1]]].isalnum():
            cur.append(i)
        else:
            if cur: groups.append(cur)
            cur = [i]
    if cur: groups.append(cur)
    return groups

def decode_word_vote(lg, f, id2tag):
    probs = torch.softmax(lg.float(), -1)
    st, en, text = f["starts"], f["ends"], f["text"]
    words = []
    for g in word_groups(st, en, text):
        p = probs[g].sum(0)
        words.append((st[g[0]], en[g[-1]], p))
    spans, cur = [], None
    for s, e, p in words:
        t = id2tag[int(p.argmax())]
        if t == "O":
            if cur: spans.append(cur); cur = None
            continue
        pref = t.split("-", 1)[0]
        lab_scores = {}
        for k, name in id2tag.items():
            if name != "O":
                lab = name.split("-", 1)[1]
                lab_scores[lab] = lab_scores.get(lab, 0.0) + float(p[k])
        if cur is None or pref == "B":
            if cur: spans.append(cur)
            cur = [s, e, dict(lab_scores)]
        else:
            cur[1] = e
            for lab, v in lab_scores.items(): cur[2][lab] = cur[2].get(lab, 0.0) + v
    if cur: spans.append(cur)
    return [[s, e, max(sc, key=sc.get)] for s, e, sc in spans]

def decode_predictions(logits_by_doc, feats, id2tag, C):
    preds = {}
    for d, lg in logits_by_doc.items():
        if C["DECODE_MODE"] == "word_vote":
            preds[d] = decode_word_vote(lg, feats[d], id2tag)
        else:
            tag_ids = lg.argmax(-1).tolist()
            preds[d] = bio_to_spans(tag_ids, feats[d]["starts"], feats[d]["ends"], id2tag, mode=C["DECODE_MODE"])
    return preds

def evaluate(model, feats, gold, labels, id2tag, C, pad_id, device):
    lg = predict_logits(model, feats, C, pad_id, device)
    preds = decode_predictions(lg, feats, id2tag, C)
    sub_gold = {d: [[s, e, l] for s, e, l, _ in gold[d]] for d in feats}
    return strict_scores(sub_gold, preds, labels), preds

def class_weights(train_ex, n_tags, C, device):
    w = torch.ones(n_tags)
    if C["CLASS_WEIGHTING"] == "o_down":
        w[0] = C["O_WEIGHT"]
    elif C["CLASS_WEIGHTING"] == "inv_sqrt":
        cnt = torch.ones(n_tags)
        for e in train_ex:
            for t in e["tags"]: cnt[t] += 1
        w = (cnt.sum() / cnt).sqrt(); w = w / w.mean()
    return w.to(device)

def train(model, train_ex, val_feats, val_gold, labels, id2tag, C, pad_id, device, teacher=None, log=print):
    params = [p for p in model.parameters() if p.requires_grad]
    head_ids = {id(p) for p in model.head.parameters()}
    if model.prompt is not None: head_ids.add(id(model.prompt))
    groups = [{"params": [p for p in params if id(p) in head_ids], "lr": C["HEAD_LR"]},
              {"params": [p for p in params if id(p) not in head_ids], "lr": C["LR"]}]
    groups = [g for g in groups if g["params"]]
    opt = torch.optim.AdamW(groups, weight_decay=C["WEIGHT_DECAY"])
    steps_per_epoch = math.ceil(len(train_ex) / C["BATCH_SIZE"])
    total = steps_per_epoch * C["EPOCHS"]
    warm = int(C["WARMUP_FRAC"] * total)
    base = [g["lr"] for g in opt.param_groups]
    _amp = C["AMP"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=_amp) if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler") \
        else torch.cuda.amp.GradScaler(enabled=_amp)
    cw = class_weights(train_ex, model.head_out, C, device)
    rng = random.Random(C["SEED"])
    best, best_state, history, step, bad = -1.0, None, [], 0, 0
    t0 = time.time()
    for ep in range(1, C["EPOCHS"] + 1):
        model.train()
        order = list(range(len(train_ex))); rng.shuffle(order)
        if C["EMPTY_WINDOW_KEEP"] < 1.0:
            order = [i for i in order if any(t != 0 for t in train_ex[i]["tags"]) or rng.random() < C["EMPTY_WINDOW_KEEP"]]
        run_loss = 0.0
        for bi in range(0, len(order), C["BATCH_SIZE"]):
            batch = [train_ex[i] for i in order[bi:bi + C["BATCH_SIZE"]]]
            ids, att, tags = collate(batch, pad_id, device)
            for g, b in zip(opt.param_groups, base):
                g["lr"] = b * (step / max(1, warm) if step < warm else max(0.0, (total - step) / max(1, total - warm)))
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=C["AMP"] and device.type == "cuda"):
                logits = model(ids, att)
                loss = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), tags.reshape(-1), weight=cw, ignore_index=-100)
                if teacher is not None:
                    with torch.no_grad():
                        tl = teacher(ids, att).float()
                    m = tags.reshape(-1) != -100
                    T_ = C["KD_TEMPERATURE"]
                    kd = F.kl_div(F.log_softmax(logits.float().reshape(-1, logits.size(-1))[m] / T_, -1),
                                  F.softmax(tl.reshape(-1, tl.size(-1))[m] / T_, -1), reduction="batchmean") * T_ * T_
                    loss = (1 - C["KD_ALPHA"]) * loss + C["KD_ALPHA"] * kd
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, C["GRAD_CLIP"])
            scaler.step(opt); scaler.update()
            run_loss += loss.item(); step += 1
            if step % C["LOG_EVERY"] == 0:
                log(f"  epoch {ep} step {step}/{total} loss {run_loss / C['LOG_EVERY']:.4f} ({time.time() - t0:.0f}s)")
                run_loss = 0.0
        if val_feats:
            sc, _ = evaluate(model, val_feats, val_gold, labels, id2tag, C, pad_id, device)
            history.append({"epoch": ep, "val_macro_F1": sc["macro_F1"], "val_micro_F1": sc["micro_F1"], "time_s": time.time() - t0})
            log(f"epoch {ep}: val strict macro-F1 {sc['macro_F1']:.4f} | micro-F1 {sc['micro_F1']:.4f} ({time.time() - t0:.0f}s)")
            if sc["macro_F1"] > best:
                best, bad = sc["macro_F1"], 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                bad += 1
                if bad >= C["PATIENCE"]:
                    log(f"early stop (no improvement for {bad} epochs)"); break
    if best_state is not None:
        model.load_state_dict(best_state)
    return {"best_val_macro_F1": best, "history": history, "train_time_s": time.time() - t0, "steps": step}

### 1. Data, annotation checks, tokenisation with character offsets, alignment ceiling

In [ ]:
if SMOKE_TEST:
    labels = ["APP", "RESP", "A.COUNSEL", "R.COUNSEL", "JUDGE", "WIT", "AUTH", "COURT", "STAT", "PREC", "DATE", "CASENO"]
    _t = ("REPORTABLE IN THE SUPREME COURT OF INDIA CIVIL APPEAL NO. 1234 OF 2019 Ram Kumar ... Appellant(s) "
          "VERSUS State of U.P. ... Respondent(s) JUDGMENT Hon'ble Justice A. B. Sharma. Section 302 of the Indian Penal Code. "
          "Dated: 12.03.2019. ") * 3
    def _ents(t):
        out = []
        for needle, lab in [("SUPREME COURT OF INDIA", "COURT"), ("CIVIL APPEAL NO. 1234 OF 2019", "CASENO"), ("Ram Kumar", "APP"),
                            ("State of U.P.", "RESP"), ("A. B. Sharma", "JUDGE"), ("Section 302", "STAT"), ("Indian Penal Code", "STAT"), ("12.03.2019", "DATE")]:
            i = t.find(needle)
            while i >= 0:
                out.append((i, i + len(needle), lab, needle)); i = t.find(needle, i + 1)
        return out
    texts = {f"d{i}": _t for i in range(6)}
    gold = {d: _ents(t) for d, t in texts.items()}
    test_texts = {"t0": _t}
else:
    labels = [l.strip() for l in open(data_file("Train/labels.txt"), encoding="utf-8") if l.strip()]
    texts, gold = {}, {}
    for fold in ("fold1", "fold2"):
        texts.update(parse_texts(load_json(data_file(f"Train/{fold}-text.json"))))
        gold.update(parse_gold(load_json(data_file(f"Train/{fold}-gold.json"))))
    test_texts = parse_texts(load_json(data_file("Test/fold3-text.json")))
print(f"labels ({len(labels)}): {labels}")
print(f"train documents: {len(texts)} (with gold: {len(gold)}) | test documents: {len(test_texts)}")
_ex = next(iter(gold))
print("example gold:", _ex, gold[_ex][:3])
check_gold(texts, gold, labels)
from collections import Counter
lab_counts = Counter(l for es in gold.values() for _, _, l, _ in es)
print("entities per label:", dict(lab_counts.most_common()))
doc_chars = sorted(len(t) for t in texts.values())
print(f"document length (chars): median {doc_chars[len(doc_chars)//2]:,}, max {doc_chars[-1]:,}")

tag_list, tag2id = make_label_maps(labels)
id2tag = dict(enumerate(tag_list))
train_ids, val_ids = split_docs([d for d in texts if d in gold], VAL_FRAC, SEED)
if SMOKE_TEST: train_ids, val_ids = train_ids, train_ids[:2]

if SMOKE_TEST:
    tokenizer = LegalBPETokenizer.train([_t] * 5, vocab_size=400, verbose=False)
else:
    MODEL_PATH = find_model_dir(C["ARCH"])
    print("Phase-1 model folder:", MODEL_PATH)
    tokenizer = LegalBPETokenizer.from_pretrained(MODEL_PATH)
t0 = time.time()
train_feats, dropped_tr, n_tr = featurize(train_ids, texts, gold, tokenizer, tag2id, C)
val_feats, dropped_va, n_va = featurize(val_ids, texts, gold, tokenizer, tag2id, C)
test_feats, _, _ = featurize(list(test_texts), test_texts, None, tokenizer, tag2id, C)
print(f"tokenised in {time.time() - t0:.0f}s | train {len(train_ids)} docs, val {len(val_ids)} docs, test {len(test_feats)} docs")
print(f"entities lost by flat BIO (overlaps): train {dropped_tr}/{n_tr}, val {dropped_va}/{n_va}")
tok_lens = sorted(len(f["ids"]) for f in train_feats.values())
print(f"document length (tokens): median {tok_lens[len(tok_lens)//2]:,}, max {tok_lens[-1]:,}")

_ceil_pred = {d: bio_to_spans(f["tags"], f["starts"], f["ends"], id2tag, "strict") for d, f in val_feats.items()}
_ceil_gold = {d: [[s, e, l] for s, e, l, _ in gold[d]] for d in val_feats}
ceiling = strict_scores(_ceil_gold, _ceil_pred, labels)
print(f"ALIGNMENT CEILING on validation (perfect tagger): strict macro-F1 {ceiling['macro_F1']:.4f}, micro-F1 {ceiling['micro_F1']:.4f}")
_bad = [(d, s, e, l, texts[d][s:e]) for d in val_feats for s, e, l, _ in gold[d] if [s, e, l] not in _ceil_pred[d]][:8]
for b in _bad: print("  not recoverable:", b)

_wx = window_examples(train_feats, C)
print(f"training windows: {len(_wx)} (MAX_LEN {MAX_LEN}, stride {STRIDE}); "
      f"windows without entities: {sum(all(t == 0 for t in e['tags']) for e in _wx)}")

### 2. Experiment runner (train → validate → test predictions → submission.zip, metrics.json)

In [ ]:
import gc

def build_ner(c):
    arch, method = c["ARCH"], c["METHOD"]
    if SMOKE_TEST:
        cfg = dict(arch=arch, vocab_size=len(tokenizer), n_embd=32, n_layer=4, n_enc_layer=4, n_dec_layer=2, n_head=4,
                   ffn_hidden=64, rope_theta=1e4, dropout=0.0, max_positions=1344,
                   bos_token_id=tokenizer.eos_token_id, eos_token_id=tokenizer.eos_token_id)
        bb = ARCH_CLASSES[arch](cfg)
    else:
        cfg, bb = load_backbone(find_model_dir(arch), ARCH_CLASSES)
    if method == "kd":
        bb = keep_layers(arch, bb, c["KD_KEEP_LAYERS"] if not SMOKE_TEST else [0, 2])
    m = NERModel(arch, bb, len(tag_list), context=c["CONTEXT"], head=c["HEAD"],
                 n_prompt=c["N_PROMPT"] if method == "prompt" else 0)
    info = apply_method(m, method, c)
    return m.to(device), info

def load_teacher(c):
    tdir = os.path.join(WORK, c["KD_TEACHER_RUN"]) if c.get("KD_TEACHER_RUN") else c["KD_TEACHER_DIR"]
    assert tdir and os.path.exists(os.path.join(tdir, "ner_model.pt")), \
        f"KD teacher not found in {tdir}: run the teacher experiment first (METHOD full or partial saves ner_model.pt)"
    tc = json.load(open(os.path.join(tdir, "run_config.json")))
    teacher, tinfo = build_ner(tc)
    teacher.load_state_dict(torch.load(os.path.join(tdir, "ner_model.pt"), map_location="cpu"))
    teacher.eval()
    for p in teacher.parameters(): p.requires_grad = False
    print(f"teacher: {tc['RUN_NAME']} ({tinfo['total_params'] / 1e6:.1f}M params)")
    return teacher, tc["RUN_NAME"]

def run_experiment(c):
    random.seed(c["SEED"]); np.random.seed(c["SEED"]); torch.manual_seed(c["SEED"])
    out_dir = os.path.join(WORK, c["RUN_NAME"]); os.makedirs(out_dir, exist_ok=True)
    pad = tokenizer.eos_token_id
    model, info = build_ner(c)
    print(f"{c['ARCH']} + {c['METHOD']} (context {c['CONTEXT']}, lr {c['LR']}): "
          f"trainable {info['trainable_params'] / 1e6:.2f}M / {info['total_params'] / 1e6:.2f}M ({info['trainable_pct']:.2f}%)")
    teacher, teacher_name = (load_teacher(c) if c["METHOD"] == "kd" else (None, None))
    train_ex = window_examples(train_feats, c)
    if device.type == "cuda": torch.cuda.reset_peak_memory_stats()
    result = train(model, train_ex, val_feats, gold, labels, id2tag, c, pad, device, teacher=teacher)
    peak = torch.cuda.max_memory_allocated() / 2**30 if device.type == "cuda" else 0.0
    del teacher

    t0 = time.time()
    val_logits = predict_logits(model, val_feats, c, pad, device)
    val_s = time.time() - t0
    _gold_val = {d: [[s, e, l] for s, e, l, _ in gold[d]] for d in val_feats}
    decoder_ablation = {}
    for mode in c["REPORT_DECODE_MODES"]:
        _p = decode_predictions(val_logits, val_feats, id2tag, dict(c, DECODE_MODE=mode))
        decoder_ablation[mode] = strict_scores(_gold_val, _p, labels)["macro_F1"]
    val_preds = decode_predictions(val_logits, val_feats, id2tag, c)
    val_scores = strict_scores(_gold_val, val_preds, labels)
    print("span-decoding ablation (val strict macro-F1):", {k: round(v, 4) for k, v in decoder_ablation.items()})
    errors, confusions = error_breakdown({d: gold[d] for d in val_feats}, val_preds)
    print(f"\nVALIDATION {c['RUN_NAME']}: strict macro-F1 {val_scores['macro_F1']:.4f} "
          f"(P {val_scores['macro_P']:.4f}, R {val_scores['macro_R']:.4f}) | micro-F1 {val_scores['micro_F1']:.4f} "
          f"| alignment ceiling {ceiling['macro_F1']:.4f}")
    print(f"{'label':10s} {'P':>6s} {'R':>6s} {'F1':>6s} {'gold':>6s} {'pred':>6s}")
    for l, v in sorted(val_scores["per_label"].items(), key=lambda x: -x[1]["F1"]):
        print(f"{l:10s} {v['P']:6.3f} {v['R']:6.3f} {v['F1']:6.3f} {v['support']:6d} {v['pred']:6d}")
    print("error breakdown:", errors)
    print("label confusions (gold->pred, same span):", confusions)

    t0 = time.time()
    test_preds = decode_predictions(predict_logits(model, test_feats, c, pad, device), test_feats, id2tag, c)
    test_s = time.time() - t0
    prediction = {d: [[int(s), int(e), l] for s, e, l in test_preds.get(d, [])] for d in test_texts}
    for d, sp in prediction.items():
        for s, e, l in sp:
            assert 0 <= s < e <= len(test_texts[d]) and test_texts[d][s:e] == test_texts[d][s:e].strip(), (d, s, e, l)
    json.dump(prediction, open(os.path.join(out_dir, "prediction.json"), "w"))
    with zipfile.ZipFile(os.path.join(out_dir, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(out_dir, "prediction.json"), "prediction.json")
    json.dump({k: v for k, v in c.items() if isinstance(v, (int, float, str, bool, list, type(None)))},
              open(os.path.join(out_dir, "run_config.json"), "w"), indent=2)
    metrics = {"run": c["RUN_NAME"], "arch": c["ARCH"], "method": c["METHOD"], "context": c["CONTEXT"], "head": c["HEAD"],
               "lr": c["LR"], "teacher": teacher_name,
               "val_strict_macro_F1": val_scores["macro_F1"], "val_strict_macro_P": val_scores["macro_P"],
               "val_strict_macro_R": val_scores["macro_R"], "val_strict_micro_F1": val_scores["micro_F1"],
               "alignment_ceiling_macro_F1": ceiling["macro_F1"], "decode_mode": c["DECODE_MODE"], "decoder_ablation": decoder_ablation, "per_label": val_scores["per_label"],
               "error_breakdown": errors, "label_confusions": confusions, **info,
               "train_time_s": result["train_time_s"], "epochs_run": len(result["history"]), "history": result["history"],
               "peak_gpu_GB": peak, "val_inference_s": val_s, "test_inference_s": test_s,
               "n_test_entities": sum(len(v) for v in prediction.values())}
    json.dump(metrics, open(os.path.join(out_dir, "metrics.json"), "w"), indent=2)
    json.dump({d: [[int(s), int(e), l] for s, e, l in v] for d, v in val_preds.items()},
              open(os.path.join(out_dir, "val_predictions.json"), "w"))
    teacher_needed = any(e.get("KD_TEACHER_RUN") == c["RUN_NAME"] for e in EXPERIMENTS)
    if c["METHOD"] in ("full", "partial") or c["SAVE_WEIGHTS"] or teacher_needed:
        torch.save(model.state_dict(), os.path.join(out_dir, "ner_model.pt"))
    print(f"saved {out_dir}: {sorted(os.listdir(out_dir))} | test entities {metrics['n_test_entities']} | "
          f"train {result['train_time_s'] / 60:.1f} min | peak GPU {peak:.2f} GB")
    del model; gc.collect()
    if device.type == "cuda": torch.cuda.empty_cache()
    return metrics

### 3. Run every experiment in `EXPERIMENTS` and print the comparison table

In [ ]:
import traceback
run_start, summary = time.time(), []
for i, e in enumerate(EXPERIMENTS, 1):
    c = experiment_config(e)
    used_h = (time.time() - run_start) / 3600
    longest_h = max([m["train_time_s"] for m in summary if "train_time_s" in m] or [0]) / 3600
    if used_h + 1.2 * longest_h > TIME_BUDGET_HOURS:
        print(f"SKIP {c['RUN_NAME']}: {used_h:.1f} h used, would exceed TIME_BUDGET_HOURS"); summary.append({"run": c["RUN_NAME"], "status": "skipped"}); continue
    print("\n" + "=" * 100 + f"\nEXPERIMENT {i}/{len(EXPERIMENTS)}: {c['RUN_NAME']}   ({used_h:.2f} h elapsed)\n" + "=" * 100)
    try:
        summary.append(dict(run_experiment(c), status="ok"))
    except Exception as ex:
        traceback.print_exc()
        summary.append({"run": c["RUN_NAME"], "status": f"FAILED: {type(ex).__name__}: {ex}"})
        gc.collect()
        if device.type == "cuda": torch.cuda.empty_cache()

print("\n" + "=" * 100 + "\nSUMMARY (validation = held-out training documents, strict span + label match)")
print(f"{'run':26s} {'macro-F1':>8s} {'P':>6s} {'R':>6s} {'micro':>6s} {'trainable':>10s} {'total':>8s} {'train min':>9s} {'GPU GB':>6s}")
for m in summary:
    if m.get("status") != "ok":
        print(f"{m['run']:26s} {m['status']}"); continue
    print(f"{m['run']:26s} {m['val_strict_macro_F1']:8.4f} {m['val_strict_macro_P']:6.3f} {m['val_strict_macro_R']:6.3f} "
          f"{m['val_strict_micro_F1']:6.3f} {m['trainable_params'] / 1e6:9.2f}M {m['total_params'] / 1e6:7.1f}M "
          f"{m['train_time_s'] / 60:9.1f} {m['peak_gpu_GB']:6.2f}")
json.dump(summary, open(os.path.join(WORK, "phase2_summary.json"), "w"), indent=2)
with zipfile.ZipFile(os.path.join(WORK, "phase2_all_submissions.zip"), "w", zipfile.ZIP_DEFLATED) as z:
    z.write(os.path.join(WORK, "phase2_summary.json"), "phase2_summary.json")
    for m in summary:
        d = os.path.join(WORK, m["run"])
        for f in ("submission.zip", "metrics.json", "run_config.json", "val_predictions.json"):
            if os.path.exists(os.path.join(d, f)): z.write(os.path.join(d, f), f"{m['run']}/{f}")
print("\nDownload phase2_all_submissions.zip; upload each <run>/submission.zip to Codabench separately.")